# FlashAttention：Kernel 级 Tiling + Online Softmax

> Dao et al., 2022. 核心考点：分块矩阵乘 + online softmax + HBM/SRAM IO 分析。

## 背景
标准 Attention 需物化 N*N 矩阵 S，FlashAttention 用 tiling 把 Q/K/V 分块加载到 SRAM，
计算局部 attention，用 online softmax 增量更新，从不物化完整 S 矩阵。

## Online Softmax
维护 running max m 和 running sum l：
- m_new = max(m, row_max(block))
- l_new = l * exp(m-m_new) + sum(exp(block-m_new))
- O_new = rescale(O) + softmax(block) @ V_block

## IO 复杂度
标准: O(4N^2 d) HBM 读写。FA: O(N^2 d^2 / M) HBM 读写，M=SRAM(192KB)

## 考察点
- SRAM/HBM 层次与数据搬运
- online softmax 两步变一步推导
- tiling 三重循环：Q沿行切，K/V沿行切


In [ ]:
import torch; import math

Br, Bc = 32, 32
N, d = 128, 64
Q = torch.randn(1, N, d)
K = torch.randn(1, N, d)
V = torch.randn(1, N, d)
scale = 1.0 / math.sqrt(d)

def flash_attention(Q, K, V, Br=32, Bc=32):
    # TODO: 实现三重循环 tiling + online softmax
    # 外循环: 遍历 Q 的 block (沿 tr 切)
    # 内循环: 遍历 K/V 的 block (沿 tc 切)
    # 维护 O, m, l 增量更新
    raise NotImplementedError

# 标准 attention 参考
ref = torch.nn.functional.scaled_dot_product_attention(Q, K, V)
print("ref shape:", ref.shape)

# ===== 测试验证 =====
try:
    O = flash_attention(Q, K, V)
    assert O.shape == ref.shape
    err = (O - ref).abs().max().item()
    print(f"\u2705 FlashAttention 通过, max_err={err:.6f}")
except NotImplementedError:
    print("\u2139 待实现 tiling + online softmax")
